In [1]:
import os
import sys
import subprocess
import shutil
import numpy as np
import pandas as pd
import SimpleITK as sitk
import nibabel as nib
import warnings
from nibabel.processing import conform

warnings.filterwarnings('ignore')

In [ ]:
class AlzheimerPipeline:
    def __init__(self, input_path, output_root, tools_config):
        self.input_path = input_path
        self.output_root = os.path.normpath(output_root)
        self.tools = tools_config
        self.device = 'cpu' 
        
        self.dirs = {
            "step1": os.path.join(self.output_root, "step1_input"),
            "step2_3": os.path.join(self.output_root, "step2_3_preprocessed"),
            "step4": os.path.join(self.output_root, "step4_segmentation"),
            "step7": os.path.join(self.output_root, "step7_stats"),
            "step8": os.path.join(self.output_root, "step8_registration")
        }
        for d in self.dirs.values():
            os.makedirs(d, exist_ok=True)

        print(f"🚀 KHỞI ĐỘNG PIPELINE ALZHEIMER (MODE: {self.device.upper()})")
        print(f"📂 Input: {self.input_path}")
        print(f"📂 Output: {self.output_root}")
        print("-" * 50)

    # BƯỚC 1: XỬ LÝ ĐẦU VÀO (Dùng SimpleITK để Bẻ thẳng não)
    def run_step1_input(self):
        print("\n[1/5] 📥 ĐANG XỬ LÝ ĐẦU VÀO (SimpleITK)...")
        output_dir = self.dirs["step1"]
        filename = os.path.basename(self.input_path)
        base_name = filename.replace(".img", "").replace(".nii.gz", "").replace(".nii", "")
        final_path = os.path.join(output_dir, base_name + "_conform.nii.gz")

        try:
            print(f" -> Đang đọc file ảnh: {filename}")
            img = sitk.ReadImage(self.input_path, sitk.sitkFloat32)
            
            # Ép về 3D
            if img.GetDimension() == 4:
                print(" ⚠️ Ảnh 4D. Đang lấy volume đầu tiên...")
                size = img.GetSize()
                img = sitk.Extract(img, (size[0], size[1], size[2], 0), (0, 0, 0, 0))
            
            # Resample về chuẩn 1mm isotropic & Orientation chuẩn
            print(" 🔄 Đang Resample & Bẻ thẳng ảnh (Conform)...")
            new_size = [256, 256, 256]
            new_spacing = [1.0, 1.0, 1.0]
            new_direction = [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0] # Identity
            
            ref_img = sitk.Image(new_size, sitk.sitkFloat32)
            ref_img.SetOrigin([0,0,0])
            ref_img.SetSpacing(new_spacing)
            ref_img.SetDirection(new_direction)
            
            tx = sitk.CenteredTransformInitializer(ref_img, img, sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.GEOMETRY)
            
            resampler = sitk.ResampleImageFilter()
            resampler.SetReferenceImage(ref_img)
            resampler.SetTransform(tx)
            resampler.SetInterpolator(sitk.sitkLinear)
            resampler.SetDefaultPixelValue(0)
            
            img_conformed = resampler.Execute(img)
            sitk.WriteImage(img_conformed, final_path)
            print(f"✅ Đã xử lý xong: {final_path}")
            return final_path

        except Exception as e:
            print(f"❌ Lỗi xử lý Step 1: {e}")
            return None

    # BƯỚC 2 & 3: HD-BET & N4 (Giữ nguyên logic lệnh String cho Windows)
    def run_step2_3_preprocess(self, input_nifti):
        print("\n[2/5] 🛠️ ĐANG CHẠY PRE-PROCESSING (HD-BET & N4)...")
        output_dir = self.dirs["step2_3"]
        filename = os.path.basename(input_nifti).replace(".nii.gz", "")
        
        bet_output = os.path.join(output_dir, f"{filename}_bet.nii.gz")
        
        if os.path.exists(bet_output):
             print(" -> Đã có file HD-BET.")
        else:
            print(" >>> Đang chạy HD-BET (Mode: CPU)...")
            cmd = f'hd-bet -i "{input_nifti}" -o "{bet_output}" -device cpu --disable_tta'
            try:
                subprocess.run(cmd, check=True, shell=True)
                print(" ✅ HD-BET hoàn tất.")
            except subprocess.CalledProcessError as e:
                print(f"❌ Lỗi HD-BET: {e}")
                shutil.copy(input_nifti, bet_output)

        n4_output = os.path.join(output_dir, f"{filename}_final_preproc.nii.gz")

        if "n4" in self.input_path.lower(): 
            shutil.copy(bet_output, n4_output)
        else:
            print(" >>> Đang chạy N4 Bias Correction...")
            try:
                img = sitk.ReadImage(bet_output, sitk.sitkFloat32)
                corrector = sitk.N4BiasFieldCorrectionImageFilter()
                corrector.SetMaximumNumberOfIterations([20, 20, 20])
                img_corrected = corrector.Execute(img)
                sitk.WriteImage(img_corrected, n4_output)
                print(" ✅ Xong N4.")
            except:
                shutil.copy(bet_output, n4_output)
        
        return n4_output

    # BƯỚC 4: SEGMENTATION (ĐÃ KHÔI PHỤC LOGIC STREAMING LOG CŨ)
    def run_step4_segmentation(self, input_nifti):
        print("\n[3/5] 🧠 ĐANG CHẠY FASTSURFER AI (Mode: CPU)...")
        output_dir = self.dirs["step4"]
        fastsurfer_root = self.tools['fastsurfer_root']
        script_path = os.path.join(fastsurfer_root, "FastSurferCNN", "run_prediction.py")
        
        sid = os.path.basename(input_nifti).replace(".nii.gz", "").replace("_final_preproc", "")
        mri_dir = os.path.join(output_dir, sid, "mri")
        expected_output = os.path.join(mri_dir, "aparc.DKTatlas+aseg.deep.mgz")
        
        if os.path.exists(expected_output):
            print(f"✅ Đã có kết quả Segmentation: {expected_output}")
            return expected_output

        # Config files check (Giống code cũ)
        cfg_dir = os.path.join(fastsurfer_root, "FastSurferCNN", "config")
        ckpt_dir = os.path.join(fastsurfer_root, "checkpoints")
        
        cmd = [
            sys.executable, script_path,
            "--t1", input_nifti,
            "--sd", output_dir,
            "--sid", sid,
            "--device", "cpu",
            "--viewagg_device", "cpu",
            "--batch_size", "1"
        ]
        
        my_env = os.environ.copy()
        my_env["PYTHONPATH"] = fastsurfer_root + os.pathsep + my_env.get("PYTHONPATH", "")

        try:
            print(f" >>> Đang xử lý Subject: {sid}...")
            
            process = subprocess.Popen(cmd, env=my_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                print(line.strip())            
            process.wait()
            
            if process.returncode != 0:
                 print("❌ Lỗi FastSurfer (Process returned non-zero code)")
                 return None

            if os.path.exists(mri_dir):
                if os.path.exists(expected_output):
                    print(f" ✅ Segmentation thành công: {expected_output}")
                    return expected_output
                
                # Fallback tìm file mgz khác
                found = [f for f in os.listdir(mri_dir) if f.endswith(".mgz")]
                if found: 
                    print(f" ✅ Tìm thấy file output: {found[0]}")
                    return os.path.join(mri_dir, found[0])
            
            print(f"❌ Không tìm thấy output tại: {mri_dir}")
            return None

        except Exception as e:
            print(f"❌ Lỗi gọi subprocess: {e}")
            return None
        

In [3]:
BASE_DIR = r"/mnt/c/Users/ADMIN/Desktop/MRI/alzheimer"
input_file_path = r"/home/trandangduat/freesurfer-test/disc1/OAS1_0001_MR1/PROCESSED/MPRAGE/T88_111/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc.img"

CONFIG = {
    "INPUT_DATA": input_file_path, 
    "OUTPUT_ROOT": os.path.join(BASE_DIR, "final_r"),
    "TOOLS": {
        "fastsurfer_root": os.path.join(BASE_DIR, "tools", "FastSurfer"),
        "vox2cortex_root": os.path.join(BASE_DIR, "tools", "Vox2Cortex")
    }
}

print(f"Project Root: {BASE_DIR}")
print(f"Input File: {CONFIG['INPUT_DATA']}")
print(f"Output Dir: {CONFIG['OUTPUT_ROOT']}")

Project Root: /mnt/c/Users/ADMIN/Desktop/MRI/alzheimer
Input File: /home/trandangduat/freesurfer-test/disc1/OAS1_0001_MR1/PROCESSED/MPRAGE/T88_111/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc.img
Output Dir: /mnt/c/Users/ADMIN/Desktop/MRI/alzheimer/final_r


In [4]:
pipeline = AlzheimerPipeline(
    input_path=CONFIG["INPUT_DATA"],
    output_root=CONFIG["OUTPUT_ROOT"],
    tools_config=CONFIG["TOOLS"]
)

🚀 KHỞI ĐỘNG PIPELINE ALZHEIMER (MODE: CPU)
📂 Input: /home/trandangduat/freesurfer-test/disc1/OAS1_0001_MR1/PROCESSED/MPRAGE/T88_111/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc.img
📂 Output: /mnt/c/Users/ADMIN/Desktop/MRI/alzheimer/final_r
--------------------------------------------------


In [5]:
try:
    nifti_file = pipeline.run_step1_input()
    
    if nifti_file:
        preproc_file = pipeline.run_step2_3_preprocess(nifti_file)
        
        if preproc_file:
            seg_file = pipeline.run_step4_segmentation(preproc_file)
            if seg_file:
                # preproc_file đóng vai trò là 'norm_file' (ảnh cấu trúc để tính intensity)
                stats_path = pipeline.run_step7_full_stats(seg_file=seg_file, norm_file=preproc_file)
                
                # Gọi bước 6 Mesh (như đã làm trước đó)
                pipeline.run_step6_mesh_generation(preproc_file)
    print("\n🏁 PIPELINE HOÀN TẤT!")
except Exception as e:
    print(f"\n❌ CÓ LỖI XẢY RA: {e}")


[1/5] 📥 ĐANG XỬ LÝ ĐẦU VÀO (SimpleITK)...
 -> Đang đọc file ảnh: OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc.img
 🔄 Đang Resample & Bẻ thẳng ảnh (Conform)...


NiftiImageIO (0x561a0ce792d0): /home/trandangduat/freesurfer-test/disc1/OAS1_0001_MR1/PROCESSED/MPRAGE/T88_111/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc.img is Analyze file and it's deprecated 

NiftiImageIO (0x561a0ce792d0): /home/trandangduat/freesurfer-test/disc1/OAS1_0001_MR1/PROCESSED/MPRAGE/T88_111/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc.img is Analyze file and it's deprecated 



✅ Đã xử lý xong: /mnt/c/Users/ADMIN/Desktop/MRI/alzheimer/final_r/step1_input/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc_conform.nii.gz

[2/5] 🛠️ ĐANG CHẠY PRE-PROCESSING (HD-BET & N4)...
 -> Đã có file HD-BET.

[3/5] 🧠 ĐANG CHẠY FASTSURFER AI (Mode: CPU)...
✅ Đã có kết quả Segmentation: /mnt/c/Users/ADMIN/Desktop/MRI/alzheimer/final_r/step4_segmentation/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc_conform/mri/aparc.DKTatlas+aseg.deep.mgz

[6/5] 📊 ĐANG TẠO BẢNG THỐNG KÊ CHI TIẾT...
 >>> Đang thử tính toán (Mode: Full Stats) với OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc_conform_final_preproc.nii.gz...
 ⚠️ Tool FastSurfer gặp lỗi (ZeroDivision/Mismatch). Đang chuyển sang tính thủ công...
 🔄 Đang chạy tính toán Thể tích bằng Numpy (Volume Only)...
 ✅ Đã tạo file Stats (Manual Mode): /mnt/c/Users/ADMIN/Desktop/MRI/alzheimer/final_r/step7_stats/OAS1_0001_MR1_mpr_n4_anon_111_t88_gfc_conform.stats
    --- Kết quả mẫu ---
1    2     284676     284676.0000     Region_2
2    4     37004      37004.0000      Reg